# Schedule-Free and Parameter-Free Optimisers (C04 Topic 06)

**Citations:** Armijo (1966); Polyak (1969); Barzilai & Borwein (1988); Orabona & Tommasi (2017 COCOB); Defazio & Mishchenko (2023 D-Adaptation); Mishchenko & Defazio (2023 Prodigy); Defazio et al. (2024 Schedule-Free).

### Purpose & Topic Bridges
Standard deep learning workflows require fixing total training steps $T_{\max}$ upfront to configure cosine annealing or warmup schedules. If stopped early, the model remains under-converged; if extended, the schedule expires. Furthermore, tuning base learning rates $\eta$ requires extensive trial-and-error.

**Schedule-Free Optimisation** eliminates learning rate schedules entirely via continuous iterate averaging: models can be trained indefinitely or stopped at any step while matching cosine decay performance. **Polyak Step-Size** provides exact closed-form optimal steps for known optimal loss $f^*$. **Barzilai-Borwein (`BB`)** computes non-monotonic spectral step sizes from consecutive iterations, and **D-Adaptation / Prodigy** automatically tracks the distance to the optimum $D = \|x_0 - x^*\|$ to set parameter-free learning rates.

**Decision Boundary:** Use **Schedule-Free AdamW** for perpetual pretraining, streaming data, and workflows where total epoch count is undecided. Use **Barzilai-Borwein** for convex deterministic problems. When NOT to use: Never evaluate or deploy a Schedule-Free model using the training weights $y_t$; always call `optimizer.eval()` to swap to the averaged weights $x_t$.

## 2. Mathematical Foundations & Governing Equations

1. **Schedule-Free AdamW:** Maintains base buffer $z_t$, running average $x_t$, and query point $y_t$. Given gradient $g_t = \nabla f(y_t)$:
$$m_{t+1} = \beta_1 m_t + (1-\beta_1) g_t, \quad v_{t+1} = \beta_2 v_t + (1-\beta_2) g_t^2, \quad \hat{v} = v_{t+1} / (1-\beta_2^t)$$
$$u_t = \frac{m_{t+1}}{\sqrt{\hat{v}} + \epsilon} + \lambda y_t, \quad z_{t+1} = z_t - \gamma u_t$$
$$x_{t+1} = (1 - c_t') x_t + c_t' z_{t+1}, \quad y_{t+1} = (1 - c_{t+1}) z_{t+1} + c_{t+1} x_{t+1}$$
2. **Polyak Optimal Step-Size:** $\gamma_t = \frac{f(x_t) - f^*}{\|\nabla f(x_t)\|^2 + \epsilon}$, $x_{t+1} = x_t - \gamma_t \nabla f(x_t)$. Minimizes the distance upper bound $\|x_{t+1} - x^*\|^2$.
3. **Barzilai-Borwein Spectral Step-Size (BB1):** $\gamma_k = \frac{s_{k-1}^T s_{k-1}}{s_{k-1}^T y_{k-1} + \epsilon}$, where $s = x_k - x_{k-1}$ and $y = g_k - g_{k-1}$.
4. **D-Adaptation:** Estimates $D_t \approx \|x_0 - x^*\|$, setting effective learning rate $\eta_t = \eta_0 \cdot D_t$.

In [ ]:
import sys
from pathlib import Path
import torch, torch.nn as nn

repo_root = Path.cwd().resolve()
while not (repo_root / "core").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
import core
core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = repo_root / "weights/04_training_methods_and_optimisers/01_gradient_and_curvature_optimisers/06_schedule_free_and_parameter_free_optimisers.safetensors"
weights, test_in, test_out, meta = core.parity.load_reference_checkpoint(ckpt_path, device=device)
print(f"Device: {device} | Checkpoint: {ckpt_path.name} | Loaded {len(weights)} tensors")

In [ ]:
def step_schedule_free_adamw(x, z, y, grad, m, v, lr=0.01, beta1=0.9, beta2=0.999, wd=0.01, step=1, eps=1e-8):
    m_next = beta1 * m + (1.0 - beta1) * grad
    v_next = beta2 * v + (1.0 - beta2) * (grad ** 2)
    v_hat = v_next / float(1.0 - beta2 ** step)
    u_t = (m_next / (torch.sqrt(v_hat) + eps)) + wd * y
    z_next = z - lr * u_t
    c_prime = float(1.0 / (step + 1))
    x_next = (1.0 - c_prime) * x + c_prime * z_next
    c_next = float(1.0 / (step + 2))
    y_next = (1.0 - c_next) * z_next + c_next * x_next
    return x_next, z_next, y_next, m_next, v_next

def step_polyak(param, grad, f_val, f_opt, eps=1e-8):
    gamma = (f_val - f_opt) / (torch.sum(grad ** 2) + eps)
    return param - gamma * grad

def step_barzilai_borwein(param, grad, s, y, eps=1e-8):
    gamma = torch.dot(s, s) / (torch.dot(s, y) + eps)
    return param - gamma * grad

def step_d_adaptation(param, grad, d_est, lr0=0.01, eps=1e-8):
    d_next = max(0.1, d_est + 0.05 * (float(torch.sum(torch.abs(grad))) / (param.numel()**0.5) - 1.0))
    return param - (lr0 * d_next) * (grad / (torch.norm(grad) + eps)), d_next

class LinearAnchor(nn.Module):
    def __init__(self, in_f, out_f):
        super().__init__()
        self.weight = nn.Parameter(torch.zeros(out_f, in_f))
        self.bias = nn.Parameter(torch.zeros(out_f))
    def forward(self, x):
        return torch.matmul(x, self.weight.T) + self.bias

print('ScheduleFreeUnifiedModule functions defined successfully.')

In [ ]:
x_next, z_next, y_next, _, _ = step_schedule_free_adamw(
    weights['sf_x'], weights['sf_z'], weights['sf_y'], weights['sf_grad'], weights['sf_m'], weights['sf_v'],
    lr=0.01, beta1=0.9, beta2=0.999, wd=0.01, step=1, eps=1e-8
)
err_sf = max(
    float(torch.max(torch.abs(x_next - weights['golden_sf_x']))),
    float(torch.max(torch.abs(z_next - weights['golden_sf_z']))),
    float(torch.max(torch.abs(y_next - weights['golden_sf_y'])))
)
p_polyak = step_polyak(weights['polyak_param'], weights['polyak_grad'], float(weights['f_val'][0]), float(weights['f_opt'][0]))
err_polyak = float(torch.max(torch.abs(p_polyak - weights['golden_polyak_param'])))
p_bb = step_barzilai_borwein(weights['bb_param'], weights['bb_grad'], weights['bb_s'], weights['bb_y'])
err_bb = float(torch.max(torch.abs(p_bb - weights['golden_bb_param'])))
p_dadapt, _ = step_d_adaptation(weights['d_param'], weights['d_grad'], float(weights['d_est'][0]), lr0=0.01)
err_dadapt = float(torch.max(torch.abs(p_dadapt - weights['golden_d_adapt_param'])))
anchor = LinearAnchor(8, 4)
anchor.weight.data.copy_(weights['anchor_w'].T)
anchor.bias.data.copy_(weights['anchor_b'])
anchor_out = anchor(weights['anchor_x']).detach()
err_anchor = float(torch.max(torch.abs(anchor_out - weights['golden_anchor_out'])))

for name, err in [('Schedule-Free', err_sf), ('Polyak', err_polyak), ('BB', err_bb), ('D-Adapt', err_dadapt), ('Anchor', err_anchor)]:
    assert err < 1e-5, f'{name} invariant failed: {err:.2e}'
print(f'All 5 invariants verified! SF: {err_sf:.2e}, Polyak: {err_polyak:.2e}, BB: {err_bb:.2e}, D-Adapt: {err_dadapt:.2e}, Anchor: {err_anchor:.2e}')

In [ ]:
track_fns = {
    'Linear Anchor': lambda: anchor(weights['anchor_x']),
    'Schedule-Free Step': lambda: step_schedule_free_adamw(weights['sf_x'], weights['sf_z'], weights['sf_y'], weights['sf_grad'], weights['sf_m'], weights['sf_v']),
    'Polyak Step': lambda: step_polyak(weights['polyak_param'], weights['polyak_grad'], float(weights['f_val'][0]), float(weights['f_opt'][0])),
    'Barzilai-Borwein': lambda: step_barzilai_borwein(weights['bb_param'], weights['bb_grad'], weights['bb_s'], weights['bb_y']),
    'D-Adaptation': lambda: step_d_adaptation(weights['d_param'], weights['d_grad'], float(weights['d_est'][0])),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=8)
print(core.benchmark.format_benchmark_table(bench_results))

In [ ]:
# Active Failure Mode: Evaluating Schedule-Free Models using Training Iterate y_t instead of x_t
# During training, y_t carries high stochastic gradient noise because base buffer z_t takes raw un-decayed steps.
# The smooth, converged solution is x_t (the running Polyak average). Evaluating y_t causes high variance spikes:
torch.manual_seed(123)
theta_star = torch.zeros(8, device=device)
x_curr, z_curr = torch.randn(8, device=device) * 0.05, torch.randn(8, device=device) * 0.05
y_curr = 0.5 * z_curr + 0.5 * x_curr

# Simulate 20 noisy training steps on quadratic loss f(w) = 0.5 * ||w - w*||^2 with stochastic noise:
for t in range(1, 21):
    noise = torch.randn(8, device=device) * 0.5  # high gradient noise
    grad_noisy = (y_curr - theta_star) + noise
    x_curr, z_curr, y_curr, _, _ = step_schedule_free_adamw(x_curr, z_curr, y_curr, grad_noisy, torch.zeros(8), torch.zeros(8), lr=0.05, step=t)

var_eval_y = float(torch.norm(y_curr - theta_star)**2)  # Raw training weights (noisy)
var_eval_x = float(torch.norm(x_curr - theta_star)**2)  # Averaged eval weights (smooth)
variance_ratio = var_eval_y / (var_eval_x + 1e-8)

print(f'[FAILURE DEMONSTRATED] Evaluating with raw training weights y_t error: {var_eval_y:.4f}')
print(f'[CORRECT] Calling optimizer.eval() with averaged weights x_t error: {var_eval_x:.4f} ({variance_ratio:.1f}x lower error!)')
assert var_eval_x < var_eval_y, 'Averaged iterate x_t failed to achieve lower variance than y_t'
print('[DIAGNOSTIC] Always call optimizer.eval() before model evaluation or saving checkpoints with Schedule-Free.')

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Schedule-Free Decouples Training Horizon:** By using iterate interpolation $y_t = (1-c_t)z_t + c_t x_t$, Schedule-Free AdamW eliminates cosine schedule tuning and enables perpetual pretraining on continuous data streams.
2. **Eval Mode is Mandatory:** The training iterate $y_t$ carries un-decayed noise; switching to $x_t$ via `eval()` is essential to unlock low-variance generalization.
3. **Curvature Isomorphism:** Barzilai-Borwein two-point spectral steps mirror scalar secant approximations across geophysical inverse problems and nonlinear fluid dynamics.

---
**Curriculum Navigation:** [Master Index](file:///c:/Users/Liam/Code/AI-Examples/README.md) | [Previous: 05_second_order_and_curvature_optimisers](file:///c:/Users/Liam/Code/AI-Examples/notebooks/04_training_methods_and_optimisers/01_gradient_and_curvature_optimisers/05_second_order_and_curvature_optimisers.ipynb) | [Next: 07_learning_rate_schedulers_and_lr_finder](file:///c:/Users/Liam/Code/AI-Examples/notebooks/04_training_methods_and_optimisers/01_gradient_and_curvature_optimisers/07_learning_rate_schedulers_and_lr_finder.ipynb)